In [1]:
import json
from typing import *
from loader import load_training_problem, list_training_problems

data = list_training_problems()
problem_id = data[0]
example = load_training_problem(problem_id)

In [2]:
PROMPT_V2 = (
    "Solve task {task_id}\n\n"
    "INPUT:\n{input}\n"
    "OUTPUT PLACEHOLDER:\n{placeholder}\n"
    "OUTPUT:"
)

In [3]:
import numpy as np

grid_placeholder = np.zeros((3,3)).astype(int)
placeholder_rows = "\n".join([" ".join(str(c) for c in row) for row in grid_placeholder])

In [4]:
def grid_to_row_strings(grid: List[List[int]]) -> List[str]:
    """
    Convert a grid (list of lists) to row-string format.
    
    Args:
        grid: Grid as list of lists of integers
        
    Returns:
        List of strings, where each string represents a row
    """
    return [' '.join(map(str, row)) for row in grid]

def _format_single_prompt(input_grid: List[List[int]], placeholder_rows: str, task_id: str) -> str:
    """Format a single-input prompt with PROMPT_V2."""
    input_str = "\n".join(grid_to_row_strings(input_grid))
    return PROMPT_V2.format(task_id=task_id, input=input_str, placeholder=placeholder_rows)

In [5]:
prompt_list = []
output_list = []
for task_id in data:
    problem = load_training_problem(problem_id)
    for sample in problem["train"]:
        formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
        formatted_output = grid_to_row_strings(sample["output"])
        prompt_list.append(formatted_prompt)
        output_list.append(formatted_output)
    for sample in problem["test"]:
        formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
        formatted_output = grid_to_row_strings(sample["output"])
        prompt_list.append(formatted_prompt)
        output_list.append(formatted_output)        

In [6]:
import tiktoken

# Initialize tokenizer (using cl100k_base which is used by GPT-4)
tokenizer = tiktoken.get_encoding("cl100k_base")

# Count tokens for prompts
prompt_token_counts = [len(tokenizer.encode(prompt)) for prompt in prompt_list]

# Count tokens for outputs (join list of strings first)
output_token_counts = [len(tokenizer.encode("\n".join(output))) for output in output_list]

# Calculate statistics for prompts
prompt_stats = {
    "mean": np.mean(prompt_token_counts),
    "median": np.median(prompt_token_counts),
    "max": np.max(prompt_token_counts),
    "min": np.min(prompt_token_counts)
}

# Calculate statistics for outputs
output_stats = {
    "mean": np.mean(output_token_counts),
    "median": np.median(output_token_counts),
    "max": np.max(output_token_counts),
    "min": np.min(output_token_counts)
}

print("Prompt token statistics:")
for stat, value in prompt_stats.items():
    print(f"  {stat}: {value:.2f}")

print("\nOutput token statistics:")
for stat, value in output_stats.items():
    print(f"  {stat}: {value:.2f}")


Prompt token statistics:
  mean: 756.00
  median: 756.00
  max: 1136.00
  min: 376.00

Output token statistics:
  mean: 721.00
  median: 721.00
  max: 1101.00
  min: 341.00


In [7]:
#from vllm import LLM, SamplingParams
#sampling_params = SamplingParams(temperature=0.8, top_p=0.95)
#llm = LLM(model="Qwen/Qwen3-4B-Instruct-2507-FP8")

#outputs = llm.generate([prompt_list[0]], sampling_params)

#for output in outputs:
#    prompt = output.prompt
#    generated_text = output.outputs[0].text
#    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [8]:
train_problems = {"conversations":[]}
test_problems = {"conversations":[]}
problem = load_training_problem(data[0])
for sample in problem["train"]:
    formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
    formatted_output = "\n".join(grid_to_row_strings(sample["output"]))
    train_problem = []
    user_content = {"role":"user", "content":""}
    user_content["content"] = formatted_prompt
    assistant_content = {"role":"assistant", "content":""}
    assistant_content["content"] = formatted_output
    train_problem.append(user_content)
    train_problem.append(assistant_content)
    train_problems["conversations"].append(train_problem)
for sample in problem["test"]:
    formatted_prompt = _format_single_prompt(sample["input"], placeholder_rows, problem_id)
    formatted_output = "\n".join(grid_to_row_strings(sample["output"]))
    test_problem = []
    user_content = {"role":"user", "content":""}
    user_content["content"] = formatted_prompt
    assistant_content = {"role":"assistant", "content":""}
    assistant_content["content"] = formatted_output
    test_problem.append(user_content)
    test_problem.append(assistant_content)
    test_problems["conversations"].append(test_problem)

In [9]:
import json
with open('data.json', 'w') as f:
    json.dump(train_problems, f)

In [10]:
import unsloth
import os
import platform
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

def pick_attn_impl() -> str:
    if platform.system() == "Linux":
        try:
            import importlib
            importlib.import_module("flash_attn")
            return "flash_attention_2"
        except Exception:
            return "sdpa"
    return "sdpa"

def run_sft(
    dataset_path: str,
    output_dir: str = "qwen3_4b_singled_out_sft",
    base_model: str = "Qwen/Qwen2.5-0.5B-Instruct",
    learning_rate: float = 8e-5,
    num_train_epochs: int = 100,
    use_compile: bool = False,
):
    """Run minimal SFT on the singled-out dataset with LoRA."""

    def formatting_prompts_func(examples):
        texts = tokenizer.apply_chat_template(examples, tokenize = False, add_generation_prompt = False)
        return { "text" : texts, }
    load_dotenv()
    if os.getenv("HF_TOKEN"):
        try:
            login(os.getenv("HF_TOKEN"))
        except Exception:
            pass
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    attn_impl = pick_attn_impl()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-0.5B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
        max_seq_length = 8192,
        dtype = compute_dtype,
        load_in_4bit = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    )
    # Dataset
    with open("data.json") as f:
        raw = json.load(f)
    data = tokenizer.apply_chat_template(
        raw["conversations"],
        tokenize = False,
    )
    import pandas as pd
    data = pd.Series(data)
    data.name = "text"
    
    from datasets import Dataset
    dataset = Dataset.from_pandas(pd.DataFrame(data))
    dataset = dataset.shuffle(seed = 3407)

    # Trainer
    args = SFTConfig(
        #loss_type="dft",
        output_dir=output_dir,
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 4,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=25,
        save_steps=200,
        save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,
        optim="paged_adamw_8bit",
        ddp_find_unused_parameters=False,
        max_grad_norm=None,
    )

    from trl import SFTTrainer
    from transformers import DataCollatorForSeq2Seq
    
    trainer = SFTTrainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=8192,
    )
        
    print("[sft] Starting training...")
    trainer.train()
    print("[sft] Saving final adapter...")
    trainer.save_model(os.path.join(output_dir, "final"))
    try:
        tokenizer.save_pretrained(os.path.join(output_dir, "final"))
    except Exception:
        pass    
    return os.path.join(output_dir, "final")

/home/ubuntu/arc_back/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-11 14:28:12 [__init__.py:241] Automatically detected platform cuda.
Unsloth: Your Flash Attention 2 installation seems to be broken?
A possible explanation is you have a new CUDA version which isn't
yet compatible with FA2? Please file a ticket to Unsloth or FA2.
We shall now use Xformers instead, which does not have any performance hits!
We found this negligible impact by benchmarking on 1x A100.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
sft_path = run_sft("data.json")

==((====))==  Unsloth 2025.9.4: Fast Qwen2 patching. Transformers: 4.56.1. vLLM: 0.10.1.1.
   \\   /|    NVIDIA H100 PCIe. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 9.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.9.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 2/2 [00:00<00:00,  2.02 examples/s]


[2025-09-11 14:28:26,326] [INFO] [real_accelerator.py:260:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-11 14:28:26,831] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


[sft] Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 100 | Total steps = 100
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 70,385,664 of 564,418,432 (12.47% trained)


Step,Training Loss
25,0.197400
50,0.007700
75,0.001000
100,0.000800


Unsloth: Will smartly offload gradients to save VRAM!
[sft] Saving final adapter...


In [12]:
import re
from typing import List, Optional


def check_array(output_string: str) -> bool:
    if not output_string or not isinstance(output_string, str):
        return False
    response = output_string.strip()
    if not response:
        return False
    if '\n' in response:
        grid_match = re.search(r'[0-9\n\s]+', response)
        if not grid_match:
            return False
        grid_str = grid_match.group()
        try:
            rows = grid_str.split('\n')
            if not rows:
                return False
            grid = []
            expected_width = None
            for row in rows:
                if not row.strip():
                    return False
                parts = row.strip().split()
                if len(parts) > 1:
                    try:
                        grid_row = [int(p) for p in parts if p.strip()]
                    except ValueError:
                        return False
                else:
                    if not row.strip().isdigit():
                        return False
                    grid_row = [int(char) for char in row.strip()]
                if any(digit < 0 or digit > 9 for digit in grid_row):
                    return False
                if expected_width is None:
                    expected_width = len(grid_row)
                elif len(grid_row) != expected_width:
                    return False
                grid.append(grid_row)
            return len(grid) > 0 and len(grid[0]) > 0
        except (ValueError, IndexError):
            return False
    return False


def check_value(output_string: str, expected_value: List[List[int]]) -> bool:
    if not isinstance(expected_value, list) or not expected_value:
        return False
    if not check_array(output_string):
        return False
    parsed_grid = parse_grid_from_string(output_string)
    if parsed_grid is None:
        return False
    return parsed_grid == expected_value

def same_shape(a: List[List[int]], b: List[List[int]]) -> bool:
    if not a or not b:
        return False
    if len(a) != len(b):
        return False
    return all(len(ra) == len(rb) for ra, rb in zip(a, b))


def parse_grid_from_string(output_string: str) -> Optional[List[List[int]]]:
    if not output_string or not isinstance(output_string, str):
        return None
    response = output_string.strip()
    if not response:
        return None
    if '\n' in response:
        grid_match = re.search(r'[0-9\n\s]+', response)
        if not grid_match:
            return None
        grid_str = grid_match.group()
        try:
            rows = grid_str.split('\n')
            grid = []
            for row in rows:
                if not row.strip():
                    continue
                parts = row.strip().split()
                if len(parts) > 1:
                    try:
                        grid_row = [int(p) for p in parts if p.strip()]
                    except ValueError:
                        return None
                else:
                    if not row.strip().isdigit():
                        return None
                    grid_row = [int(char) for char in row.strip()]
                if any(digit < 0 or digit > 9 for digit in grid_row):
                    return None
                grid.append(grid_row)
            return grid if grid else None
        except (ValueError, IndexError):
            return None
    return None

def reward_function(
    completions: List[str], 
    expected_output: List[str], 
    **kwargs: Any
) -> List[float]:
    rewards = []
    for completion, expected in zip(completions, expected_output, strict=False):
        if not check_array(completion):
            rewards.append(-1.0)
            continue
        if check_value(completion, parse_grid_from_string(expected)):
            rewards.append(1.0)
        else:
            rewards.append(-0.5)
    return rewards

def reward_function_diff(
    completions: List[str],
    expected_output: List[str],
    **kwargs: Any
) -> List[float]:
    """
    For each (completion, expected) pair:
      - If either string is not a valid grid -> reward = -1.0
      - If grid shapes differ -> reward = -1.0
      - Otherwise -> reward = (# cells that differ) / (rows * cols)
    """
    rewards: List[float] = []
    for completion, expected in zip(completions, expected_output, strict=False):
        if not check_array(completion) or not check_array(expected):
            rewards.append(-1.0)
            continue

        comp_grid = parse_grid_from_string(completion)
        exp_grid = parse_grid_from_string(expected)
        if comp_grid is None or exp_grid is None or not same_shape(comp_grid, exp_grid):
            rewards.append(-1.0)
            continue

        rows = len(exp_grid)
        cols = len(exp_grid[0]) if rows else 0
        if rows == 0 or cols == 0:
            rewards.append(-0.5)
            continue

        diffs = 0
        for r in range(rows):
            for c in range(cols):
                if comp_grid[r][c] != exp_grid[r][c]:
                    diffs += 1

        rewards.append(1 - diffs / (rows * cols))
    return rewards

In [13]:
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_4b_singled_out_sft/final", # or choose "unsloth/Llama-3.2-1B-Instruct"
        max_seq_length = 8192,
        dtype = torch.bfloat16,
        load_in_4bit = True,
    )

model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
)

with open("data.json") as f:
        raw = json.load(f)
sample_data = raw["conversations"][0][0]["content"]
messages = [
            {"role": "user", "content": sample_data},
]
from unsloth.chat_templates import get_chat_template
FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
        ).to("cuda")
outputs = model.generate(input_ids = inputs, max_new_tokens = 4096, use_cache = True)
generated_tokens = outputs[:, inputs.shape[-1]:]
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(decoded[0])
print(check_array(decoded[0]))
print(check_value(decoded[0], parse_grid_from_string(raw["conversations"][0][1]["content"])))
print(check_value(raw["conversations"][0][1]["content"], parse_grid_from_string(raw["conversations"][0][1]["content"])))
print(reward_function(decoded, [raw["conversations"][0][1]["content"]]))


==((====))==  Unsloth 2025.9.4: Fast Qwen3 patching. Transformers: 4.56.1. vLLM: 0.10.1.1.
   \\   /|    NVIDIA H100 PCIe. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 9.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Already have LoRA adapters! We shall skip this step.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 7 0 0 0 4 0 0 0 0 0 7 0 0 0
0 0 0 2 0 0 0 0 0 4 0 0 0 2 0 0 0 0 0
0 0 2 0 0 0 0 0 0 4 0 0 2 0 0 0 0 0 0
0 3 0 0 0 3 0 0 0 4 0 3 0 0 0 3 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 8 7 0 0 0 0 4 0 0 0 8 7 0 0 0 0
0 0 0 0 8 0 0 3 0 4 0 0 0 0 8 0 0 3 0
0 7 0 0 0 0 0 0 0 4 0 7 0 0 0 0 0 0 0
4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 7 0 0 0 4 0 0 0 0 0 7 0 0 0
0 0 0 2 0 0 0 0 0 4 0 0 0 2 0 0 0 0 0
0 0 2 0 0 0 0 0 0 4 0 0 2 0 0 0 0 0 0
0 3 0 0 0 3 0 0 0 4 0 3 0 0 0 3 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 8 7 0 0 0 0 4 0 0 0 8 7 0 0 0 0
0 0 0 0 8 0 0 3 0 4 0 0 0 0 8 0 0 3 0
0 7 0 0 0 0 0 0 0 4 0 7 0 0 0 0 0 0 0
True
True
True
[1.0]


In [17]:
def reward_function(
    completions: List[str], 
    expected_output: List[str], 
    **kwargs: Any
) -> List[float]:
    rewards = []
    for completion, expected in zip(completions, expected_output, strict=False):
        value = completion[0]["content"]
        if not check_array(value):
            rewards.append(-1.0)
            continue
        if check_value(value, parse_grid_from_string(expected)):
            rewards.append(1.0)
        else:
            rewards.append(-0.5)
    return rewards

def reward_function_diff(
    completions: List[str],
    expected_output: List[str],
    **kwargs: Any
) -> List[float]:
    rewards: List[float] = []
    for completion, expected in zip(completions, expected_output, strict=False):
        value = completion[0]["content"]
        if not check_array(value):
            rewards.append(-1.0)
            continue
        comp_grid = parse_grid_from_string(value)
        exp_grid = parse_grid_from_string(expected)
        if comp_grid is None or exp_grid is None or not same_shape(comp_grid, exp_grid):
            rewards.append(-0.5)
            continue
        rows = len(exp_grid)
        cols = len(exp_grid[0]) if rows else 0
        diffs = 0
        for r in range(rows):
            for c in range(cols):
                if comp_grid[r][c] != exp_grid[r][c]:
                    diffs += 1
        if diffs != 0:
            rewards.append(0.5 * (1 - diffs / (rows * cols)))
        else:
            rewards.append(1)
    return rewards

def convert_conversations(raw_json):
    result = []
    for convo in raw_json["conversations"]:
        # Expecting [ {"role":"user"}, {"role":"assistant"} ]
        user_msg = convo[0]["content"]
        assistant_msg = convo[1]["content"]
        result.append({
            "prompt": [
                {"role": "user", "content": user_msg}
            ],
            "expected_output": assistant_msg
        })
    return result

def run_rl(
    #base_model: str,
    #lora_path: str,
    #dataset_path: str,
    output_dir: str = "qwen3_4b_singled_out_rl",
    learning_rate: float = 1e-5,
    num_train_epochs: int = 1,
    grad_accum: int = 4,
    num_generations: int = 4,
):
    """Run minimal GRPO on top of SFT LoRA using the same dataset."""
    import platform
    import torch
    from datasets import load_dataset, Dataset, concatenate_datasets
    from peft import PeftModel
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from trl import GRPOConfig, GRPOTrainer
    from unsloth import FastLanguageModel
    import torch
    max_seq_length = 8192 # Can increase for longer reasoning traces
    lora_rank = 128 # Larger rank = smarter, but slower
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_4b_singled_out_sft/final",
        max_seq_length = max_seq_length,
        load_in_4bit = False, # False for LoRA 16bit
        fast_inference = True, # Enable vLLM fast inference
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.2, # Reduce if out of memory
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    )
    with open("data.json") as f:
        raw = json.load(f)
    converted = convert_conversations(raw)
    dataset = Dataset.from_list(converted)  
    repeats = 10  # makes it length 4; use 4 or 8 if you want more steps per epoch
    dataset = concatenate_datasets([dataset] * repeats)
    print(dataset)
    print("num examples:", len(dataset))  # should be > 0
    from vllm import SamplingParams
    vllm_sampling_params = SamplingParams(
        min_p = 0.1,
        top_p = 1.0,
        top_k = -1,
        seed = 3407,
        stop = [tokenizer.eos_token],
        include_stop_str_in_output = True,
    )
    
    from trl import GRPOConfig, GRPOTrainer
    training_args = GRPOConfig(
        vllm_sampling_params = vllm_sampling_params,
        importance_sampling_level="sequence",
        loss_type="grpo",
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=grad_accum,
        beta=0.04,
        epsilon=3e-4,
        #num_train_epochs=num_train_epochs,
        max_steps=200,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=200,
        optim="paged_adamw_8bit",
        report_to="none",
        num_generations=4,
        max_prompt_length=4096,
        max_completion_length=2048,
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
    )
    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [
            reward_function_diff
        ],
        args = training_args,
        train_dataset = dataset,
    
        # For optional training + evaluation
        # train_dataset = new_dataset["train"],
        # eval_dataset = new_dataset["test"],
    )
    trainer.train()
    trainer.save_model(os.path.join(output_dir, "final"))
    try:
        tokenizer.save_pretrained(os.path.join(output_dir, "final"))
    except Exception:
        pass
    return os.path.join(output_dir, "final")

In [18]:
run_rl()

Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.9.4: Fast Qwen3 patching. Transformers: 4.56.1. vLLM: 0.10.1.1.
   \\   /|    NVIDIA H100 PCIe. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 9.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-0.5b-instruct with actual GPU utilization = 15.6%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.19 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 8192. Num Sequences = 224.
Unsloth: vLLM's KV Cache can use up to 11.3 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
INFO 09-11 14:32:10 [utils.py:326] non-default args: {

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  4.19it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  4.16it/s]


INFO 09-11 14:32:14 [default_loader.py:262] Loading weights took 0.26 seconds


INFO 09-11 14:32:15 [gpu_model_runner.py:2007] Model loading took 1.0560 GiB and 0.589730 seconds
INFO 09-11 14:32:24 [backends.py:548] Using cache directory: /home/ubuntu/.cache/vllm/torch_compile_cache/5d82fa4fb3/rank_0_0/backbone for vLLM's torch.compile
INFO 09-11 14:32:24 [backends.py:559] Dynamo bytecode transform time: 8.26 s
INFO 09-11 14:32:30 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 5.145 s
INFO 09-11 14:32:31 [monitor.py:34] torch.compile takes 8.26 s in total
INFO 09-11 14:32:32 [gpu_worker.py:276] Available KV cache memory: 10.07 GiB
INFO 09-11 14:32:32 [kv_cache_utils.py:849] GPU KV cache size: 879,648 tokens
INFO 09-11 14:32:32 [kv_cache_utils.py:853] Maximum concurrency for 8,192 tokens per request: 107.38x
INFO 09-11 14:32:32 [vllm_utils.py:667] Unsloth: Running patched vLLM v1 `capture_model`.
INFO 09-11 14:32:32 [vllm_utils.py:667] Unsloth: Running patched vLLM v1 `capture_model`.


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 59/59 [00:06<00:00,  8.65it/s]

...

Unsloth: Already have LoRA adapters! We shall skip this step.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151654}.


Dataset({
    features: ['prompt', 'expected_output'],
    num_rows: 20
})
num examples: 20
Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 40 | Total steps = 200
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 70,385,664 of 564,418,432 (12.47% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / reward_function_diff / mean,rewards / reward_function_diff / std
10,0.010700,-0.025107,0.162956,1070.618750,338.400000,2048.000000,0.393750,444.813574,338.400000,929.300000,0,0,0,0,0,0.066806,-0.025107,0.922953
20,0.009600,0.377688,0.243421,588.618750,338.400000,824.100000,0.050000,513.131250,338.400000,694.000000,No Log,No Log,No Log,No Log,No Log,0.060063,0.377688,0.560341
30,0.006000,0.599826,0.203137,526.031250,342.900000,687.800000,0.000000,526.031250,342.900000,687.800000,No Log,No Log,No Log,No Log,No Log,0.037596,0.599826,0.417037
40,0.006000,0.729224,0.000000,532.000000,342.000000,722.000000,0.000000,532.000000,342.000000,722.000000,No Log,No Log,No Log,No Log,No Log,0.037294,0.729224,0.264669
50,0.003900,0.729224,0.000000,532.000000,342.000000,722.000000,0.000000,532.000000,342.000000,722.000000,No Log,No Log,No Log,No Log,No Log,0.024439,0.729224,0.264669
60,0.002700,0.729224,0.000000,532.000000,380.000000,722.000000,0.000000,532.000000,380.000000,722.000000,No Log,No Log,No Log,No Log,No Log,0.016635,0.729224,0.236704
70,0.002000,0.729224,0.000000,532.000000,380.000000,684.000000,0.000000,532.000000,380.000000,684.000000,No Log,No Log,No Log,No Log,No Log,0.012219,0.729224,0.208738
80,0.001200,0.716724,0.025000,529.900000,308.400000,722.000000,0.000000,529.900000,308.400000,722.000000,No Log,No Log,No Log,No Log,No Log,0.007215,0.716724,0.287300
90,0.010900,0.713563,0.031323,529.900000,346.400000,722.000000,0.000000,529.900000,346.400000,722.000000,No Log,No Log,No Log,No Log,No Log,0.068220,0.713563,0.258926
100,0.001300,0.729224,0.000000,532.000000,342.000000,722.000000,0.000000,532.000000,342.000000,722.000000,No Log,No Log,No Log,No Log,No Log,0.008324,0.729224,0.264669


'qwen3_4b_singled_out_rl/final'

In [20]:
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_4b_singled_out_rl/final", # or choose "unsloth/Llama-3.2-1B-Instruct"
        max_seq_length = 8192,
        dtype = torch.bfloat16,
        load_in_4bit = True,
    )

model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
)

with open("data.json") as f:
        raw = json.load(f)
sample_data = raw["conversations"][0][0]["content"]
messages = [
            {"role": "user", "content": sample_data},
]
from unsloth.chat_templates import get_chat_template
FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
        ).to("cuda")
outputs = model.generate(input_ids = inputs, max_new_tokens = 4096, use_cache = True)
generated_tokens = outputs[:, inputs.shape[-1]:]
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(decoded[0])
print(check_array(decoded[0]))
print(check_value(decoded[0], parse_grid_from_string(raw["conversations"][0][1]["content"])))
print(check_value(raw["conversations"][0][1]["content"], parse_grid_from_string(raw["conversations"][0][1]["content"])))

==((====))==  Unsloth 2025.9.4: Fast Qwen3 patching. Transformers: 4.56.1. vLLM: 0.10.1.1.
   \\   /|    NVIDIA H100 PCIe. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 9.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Already have LoRA adapters! We shall skip this step.


0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 7 0 0 0 4 0 0 0 0 0 7 0 0 0
0 0 0 2 0 0 0 0 0 4 0 0 0 2 0 0 0 0 0
0 0 2 0 0 0 0 0 0 4 0 0 2 0 0 0 0 0 0
0 3 0 0 0 3 0 0 0 4 0 3 0 0 0 3 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 8 7 0 0 0 0 4 0 0 0 8 7 0 0 0 0
0 0 0 0 8 0 0 3 0 4 0 0 0 0 8 0 0 3 0
0 7 0 0 0 0 0 0 0 4 0 7 0 0 0 0 0 0 0
4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 7 0 0 0 4 0 0 0 0 0 7 0 0 0
0 0 0 2 0 0 0 0 0 4 0 0 0 2 0 0 0 0 0
0 0 2 0 0 0 0 0 0 4 0 0 2 0 0 0 0 0 0
0 3 0 0 0 3 0 0 0 4 0 3 0 0 0 3 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 8 7 0 0 0 0 4 0 0 0 8 7 0 0 0 0
0 0 0 0 8 0 0 3 0 4 0 0 0 0 8 0 0 3 0
0 7 0 0 0 0 0 0 0 4 0 7 0 0 0 0 0 0 0
True
True
True


In [23]:
sample_data = raw["conversations"][1][0]["content"]
messages = [
            {"role": "user", "content": sample_data},
]
from unsloth.chat_templates import get_chat_template
FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
        ).to("cuda")
outputs = model.generate(input_ids = inputs, max_new_tokens = 4096, use_cache = True)
generated_tokens = outputs[:, inputs.shape[-1]:]
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(decoded[0])
print(check_array(decoded[0]))
print(check_value(decoded[0], parse_grid_from_string(raw["conversations"][1][1]["content"])))
print(check_value(raw["conversations"][1][1]["content"], parse_grid_from_string(raw["conversations"][1][1]["content"])))

0 0 0 0 0 0 0 0 0
0 0 0 0 0 5 0 2 0
0 0 1 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0
0 0 0 0 1 0 0 0 0
0 0 1 0 0 0 0 0 0
0 0 2 0 0 0 0 2 0
0 2 0 0 0 5 5 0 0
0 0 0 0 0 0 0 0 0
4 4 4 4 4 4 4 4 4
0 0 0 0 0 0 0 0 0
0 0 0 0 0 5 0 2 0
0 0 1 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0
0 0 0 0 1 0 0 0 0
0 0 1 0 0 0 0 0 0
0 0 2 0 0 0 0 2 0
0 2 0 0 0 5 5 0 0
0 0 0 0 0 0 0 0 0
True
True
True


In [31]:
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen3_4b_singled_out_sft/final", # or choose "unsloth/Llama-3.2-1B-Instruct"
        max_seq_length = 8192,
        dtype = torch.bfloat16,
        load_in_4bit = True,
    )

model = FastLanguageModel.get_peft_model(
        model,
        r=128,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 32,  # Best to choose alpha = rank or rank*2
        lora_dropout = 0, # Supports any, but = 0 is optimized
        bias = "none",    # Supports any, but = "none" is 
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
)

sample_data = test_problems["conversations"][0][0]["content"]
messages = [
            {"role": "user", "content": sample_data},
]
from unsloth.chat_templates import get_chat_template
FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            return_tensors = "pt",
        ).to("cuda")
outputs = model.generate(input_ids = inputs, max_new_tokens = 4096, use_cache = True)
generated_tokens = outputs[:, inputs.shape[-1]:]
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(decoded[0])
print(check_array(decoded[0]))
print(check_value(decoded[0], parse_grid_from_string(test_problems["conversations"][0][1]["content"])))
print(check_value(test_problems["conversations"][0][1]["content"], parse_grid_from_string(test_problems["conversations"][0][1]["content"])))

==((====))==  Unsloth 2025.9.4: Fast Qwen3 patching. Transformers: 4.56.1. vLLM: 0.10.1.1.
   \\   /|    NVIDIA H100 PCIe. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 9.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Already have LoRA adapters! We shall skip this step.


0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 7 0 0 0 4 0 0 0 0 0 7 0 0 0
0 0 0 2 0 0 0 0 0 4 0 0 0 2 0 0 0 0 0
0 0 2 0 0 0 0 0 0 4 0 0 2 0 0 0 0 0 0
0 3 0 0 0 3 0 0 0 4 0 3 0 0 0 3 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 8 7 0 0 0 0 4 0 0 0 8 7 0 0 0 0
0 0 0 0 8 0 0 3 0 4 0 0 0 0 8 0 0 3 0
0 7 0 0 0 0 0 0 0 4 0 7 0 0 0 0 0 0 0
4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 7 0 0 0 4 0 0 0 0 0 7 0 0 0
0 0 0 2 0 0 0 0 0 4 0 0 0 2 0 0 0 0 0
0 0 2 0 0 0 0 0 0 4 0 0 2 0 0 0 0 0 0
0 3 0 0 0 3 0 0 0 4 0 3 0 0 0 3 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 8 7 0 0 0 0 4 0 0 0 8 7 0 0 0 0
0 0 0 0 8 0 0 3 0 4 0 0 0 0 8 0 0 3 0
0 7 0 0 0 0 0 0 0 4 0 7 0 0 0 0 0 0 0
True
True
True


In [30]:
print(test_problems["conversations"][0][1]["content"])

0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 4 0 0 0 0 3 0 0 0 0 4 0 0 0 0 3 0 0 0 0
0 2 0 0 0 0 0 2 0 4 0 2 0 0 0 0 0 2 0 4 0 2 0 0 0 0 0 2 0
0 3 0 0 0 0 0 2 0 4 0 3 0 0 0 0 0 2 0 4 0 3 0 0 0 0 0 2 0
0 0 0 0 2 0 0 0 0 4 0 0 0 0 2 0 0 0 0 4 0 0 0 0 2 0 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 6 0 0 0 0 0 0 4 0 0 6 0 0 0 0 0 0 4 0 0 6 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 6 0 0 4 0 0 0 0 0 0 6 0 0 4 0 0 0 0 0 0 6 0 0
4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 4 0 0 0 0 3 0 0 0 0 4 0 0 0 0 3 0 0 0 0
0 2 0 0 0 0 0 2 0 4 0 2 0 0 0 0 0 2 0 4 0 2 0 0 0 0 0 2 0
0 3 0 0 0 0 0 2 0 4 0 3 0 0 0 0 0 2 0 4 0 3 0 0 0 0 0 2 0
0 0 0 0 2 0 0 0 0 4 0 0 0 0 2 0 0 0 0 4 0 0 0 0 2 0 0 0 0
0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
0 0 6 0 0 0 0 0 0 4 0 0 6 0 0 0 0 0 0 4 0 0 6 0 0 0 0 0 0
0 0 0 0 0 0 0 